# Graph Centrality Algorithms

## Web server interface at https://xxxx:7473

**Username: neo4j**

**Password: ommit**

**In the web server interface, run the query to return all nodes and all relationships:**

```
match (n) return n
```

In [1]:
import neo4j

import pandas as pd

from IPython.display import display

In [2]:
driver = neo4j.GraphDatabase.driver(uri="neo4j://neo4j:7687", auth=("neo4j","ucb_mids_w205"))

In [3]:
session = driver.session(database="neo4j")

In [16]:
def load_facebook_combined():
    rows = []
    
    #combined file path
    with open('/user/projects/project-3-Softwave-ui/facebook_combined.txt', "r") as f:
        for line in f:
            a, b = line.strip().split()
            rows.append({"a": int(a), "b": int(b)})

    query = """
    UNWIND $rows AS row
    MERGE (p1:Person {id: row.a})
      ON CREATE SET p1.source = 'facebook', p1.is_ego = false
    MERGE (p2:Person {id: row.b})
      ON CREATE SET p2.source = 'facebook', p2.is_ego = false
    MERGE (p1)-[:FRIENDS_WITH]-(p2)
    """

    batch_size = 1000
    for i in range(0, len(rows), batch_size):
        session.run(query, rows=rows[i:i+batch_size])


In [ ]:
    print(f"Loaded {len(rows)} edges

In [20]:
load_facebook_combined()

Loaded 88234 edges from facebook_combined.txt


In [18]:
def my_neo4j_wipe_out_database():
    "wipe out database by deleting all nodes and relationships"
    
    query = "match (node)-[relationship]->() delete node, relationship"
    session.run(query)
    
    query = "match (node) delete node"
    session.run(query)

In [19]:
my_neo4j_wipe_out_database()

In [23]:
def read_featnames(featnames_path):
    featnames = {}

    with open(featnames_path, "r") as f:
        for line in f:
            idx, name = line.strip().split(" ", 1)
            featnames[int(idx)] = name

    return featnames

In [24]:
def load_ego_network(ego_id):
    ego_id = int(ego_id)
    
    #ego file path matched on id
    edges_path = f'/user/projects/project-3-Softwave-ui/{ego_id}.edges'
    circles_path = f'/user/projects/project-3-Softwave-ui/{ego_id}.circles'
    feat_path = f'/user/projects/project-3-Softwave-ui/{ego_id}.feat'
    egofeat_path = f'/user/projects/project-3-Softwave-ui/{ego_id}.egofeat'
    featnames_path = f'/user/projects/project-3-Softwave-ui/{ego_id}.featnames'

    featnames = read_featnames(featnames_path)

    session.run("""
    MERGE (e:Person {id: $ego_id})
    ON CREATE SET e.source = 'facebook', e.is_ego = true
    SET e.is_ego = true
    """, ego_id=ego_id)

    feature_rows = [
        {
            "feature_id": f"{ego_id}_{idx}",
            "ego_id": ego_id,
            "feature_index": idx,
            "feature_name": name
        }
        for idx, name in featnames.items()
    ]

    session.run("""
    UNWIND $rows AS row
    MERGE (f:Feature {id: row.feature_id})
    ON CREATE SET
        f.ego_id = row.ego_id,
        f.feature_index = row.feature_index,
        f.feature_name = row.feature_name
    """, rows=feature_rows)

    edge_rows = []
    with open(edges_path, "r") as f:
        for line in f:
            a, b = line.strip().split()
            edge_rows.append({
                "a": int(a),
                "b": int(b),
                "ego_id": ego_id
            })

    query_edges = """
    UNWIND $rows AS row
    MERGE (p1:Person {id: row.a})
      ON CREATE SET p1.source = 'facebook', p1.is_ego = false
    MERGE (p2:Person {id: row.b})
      ON CREATE SET p2.source = 'facebook', p2.is_ego = false
    MERGE (ego:Person {id: row.ego_id})
      ON CREATE SET ego.source = 'facebook', ego.is_ego = true
    SET ego.is_ego = true
    MERGE (p1)-[:IN_EGO_NETWORK]->(ego)
    MERGE (p2)-[:IN_EGO_NETWORK]->(ego)
    MERGE (p1)-[:FRIENDS_WITH]-(p2)
    """

    batch_size = 1000
    for i in range(0, len(edge_rows), batch_size):
        session.run(query_edges, rows=edge_rows[i:i+batch_size])

    circle_rows = []
    with open(circles_path, "r") as f:
        for line in f:
            parts = line.strip().split("\t")
            circle_name = parts[0]
            members = [int(x) for x in parts[1:]]

            for member in members:
                circle_rows.append({
                    "circle_id": f"{ego_id}_{circle_name}",
                    "circle_name": circle_name,
                    "ego_id": ego_id,
                    "member_id": member
                })

    query_circles = """
    UNWIND $rows AS row
    MERGE (c:Circle {id: row.circle_id})
    ON CREATE SET c.name = row.circle_name, c.ego_id = row.ego_id
    MERGE (p:Person {id: row.member_id})
      ON CREATE SET p.source = 'facebook', p.is_ego = false
    MERGE (ego:Person {id: row.ego_id})
      ON CREATE SET ego.source = 'facebook', ego.is_ego = true
    SET ego.is_ego = true
    MERGE (p)-[:IN_EGO_NETWORK]->(ego)
    MERGE (c)-[:HAS_MEMBER]->(p)
    """

    for i in range(0, len(circle_rows), batch_size):
        session.run(query_circles, rows=circle_rows[i:i+batch_size])

    feature_rel_rows = []
    with open(feat_path, "r") as f:
        for line in f:
            vals = list(map(int, line.strip().split()))
            person_id = vals[0]
            features = vals[1:]

            for idx, value in enumerate(features):
                if value == 1:
                    feature_rel_rows.append({
                        "person_id": person_id,
                        "feature_id": f"{ego_id}_{idx}",
                        "ego_id": ego_id
                    })

    query_person_features = """
    UNWIND $rows AS row
    MERGE (p:Person {id: row.person_id})
      ON CREATE SET p.source = 'facebook', p.is_ego = false
    MERGE (ego:Person {id: row.ego_id})
      ON CREATE SET ego.source = 'facebook', ego.is_ego = true
    SET ego.is_ego = true
    MERGE (f:Feature {id: row.feature_id})
    MERGE (p)-[:IN_EGO_NETWORK]->(ego)
    MERGE (p)-[:HAS_FEATURE {ego_id: row.ego_id}]->(f)
    """

    for i in range(0, len(feature_rel_rows), batch_size):
        session.run(query_person_features, rows=feature_rel_rows[i:i+batch_size])

    with open(egofeat_path, "r") as f:
        ego_features = list(map(int, f.readline().strip().split()))

    ego_feature_rows = []
    for idx, value in enumerate(ego_features):
        if value == 1:
            ego_feature_rows.append({
                "person_id": ego_id,
                "feature_id": f"{ego_id}_{idx}",
                "ego_id": ego_id
            })

    for i in range(0, len(ego_feature_rows), batch_size):
        session.run(query_person_features, rows=ego_feature_rows[i:i+batch_size])

    print(f"Loaded ego network {ego_id}")

In [ ]:
ego_ids = [0]

for ego_id in ego_ids:
    load_ego_network(ego_id)

## DraftCode

In [ ]:
def my_neo4j_wipe_out_database():
    "wipe out database by deleting all nodes and relationships"
    
    query = "match (node)-[relationship]->() delete node, relationship"
    session.run(query)
    
    query = "match (node) delete node"
    session.run(query)

In [ ]:
def my_neo4j_run_query_pandas(query, **kwargs):
    "run a query and return the results in a pandas dataframe"
    
    result = session.run(query, **kwargs)
    
    df = pd.DataFrame([r.values() for r in result], columns=result.keys())
    
    return df

In [ ]:
def my_neo4j_nodes_relationships():
    "print all the nodes and relationships"
   
    print("-------------------------")
    print("  Nodes:")
    print("-------------------------")
    
    query = """
        MATCH (n)
        RETURN labels(n) AS labels,
               n.id AS id,
               n.name AS name,
               n.feature_name AS feature_name
        ORDER BY labels, id, name
    """
    
    df = my_neo4j_run_query_pandas(query)
    
    number_nodes = df.shape[0]
    
    display(df)
    
    print("-------------------------")
    print("  Relationships:")
    print("-------------------------")
    
    query = """
        MATCH (n1)-[r]->(n2)
        RETURN labels(n1) AS node_1_labels,
               n1.id AS node_1_id,
               type(r) AS relationship_type,
               labels(n2) AS node_2_labels,
               n2.id AS node_2_id
        ORDER BY node_1_id, node_2_id, relationship_type
    """
    
    df = my_neo4j_run_query_pandas(query)
    
    number_relationships = df.shape[0]
    
    display(df)
    
    density = (2 * number_relationships) / (number_nodes * (number_nodes - 1))
    
    print("-------------------------")
    print("  Density:", f'{density:.2f}')
    print("-------------------------")
    

## Connected Graph and load

In [ ]:
def my_create_connected_graph():
    "create the connected graph"
    
    my_neo4j_wipe_out_database()

    query = """

    CREATE
      --Query


    """

    session.run(query)

In [ ]:
my_create_connected_graph()

In [ ]:
my_neo4j_nodes_relationships()